# Episode 23: External Tools with MCP

Every tool so far has been a Python function *you* wrote. But there's a whole ecosystem of ready-made tools — file systems, GitHub, databases, search — exposed over a standard protocol.

**In this episode you'll build:** an agent that connects to an **MCP server** and uses its tools, without writing a single tool function yourself.

## What MCP is

**MCP (Model Context Protocol)** is Anthropic's standard for exposing tools to LLMs. Before MCP, every provider had its own tool schema and every integration was bespoke. MCP fixes that: a server publishes its tools once, and any MCP-aware client can discover and call them.

Think of it as a **superset of regular tools** — instead of `@tool` functions living in your code, the tools live in a separate server process and your agent connects to them.

## Two ways to connect

AG2 supports both standard wiring patterns:

| | `MCPServer` (client-side) | `MCPServerTool` (provider-side) |
|---|---|---|
| Who connects to the server | AG2 | The LLM provider |
| Who executes the tool calls | AG2 | The LLM provider |
| Works with any LLM provider | Yes | No — provider must support it |
| Supports local stdio servers | Yes | No — URL only |
| Credentials leave your infra | No | Yes — sent to the provider |

We'll use **`MCPServer`** — it works with every provider and lets us run a local server with full control.

## Install

The MCP integration is an optional extra.

In [1]:
!pip install "ag2[mcp]" -q

## A local MCP server

An MCP server is just a process that speaks the protocol over stdin/stdout. Many ship as CLIs (`npx @modelcontextprotocol/server-filesystem`, `uvx mcp-server-github`), but writing your own is a few lines with `FastMCP`.

The cell below writes a tiny **calculator server** to disk — two tools, `add` and `multiply`. (`log_level="WARNING"` just keeps its console output quiet.)

In [2]:
server_code = '''"""A tiny local MCP server exposing two arithmetic tools."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("calculator", log_level="WARNING")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the product."""
    return a * b


if __name__ == "__main__":
    mcp.run()
'''

from pathlib import Path

Path("calc_server.py").write_text(server_code)
print("wrote calc_server.py")


wrote calc_server.py


## Connect an agent to it

`MCPServer` is a **toolkit** — give it an `MCPStdioServerConfig` describing how to launch the server, and it discovers every tool the server exposes and hands them to the agent as ordinary function tools. The agent never knows the difference.

The server is launched lazily, as a subprocess, the first time a tool is needed.

In [3]:
from dotenv import load_dotenv

load_dotenv()

import sys

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import MCPServer, MCPStdioServerConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

calculator = MCPServer(
    MCPStdioServerConfig(
        command=sys.executable,            # launch the server with this Python
        args=["calc_server.py"],           # the file we just wrote
        server_label="calculator",
    )
)

agent = Agent(
    "math_bot",
    prompt="You do math. Use your tools for every calculation.",
    config=config,
    tools=[calculator],                    # an MCP toolkit, used like any tool
)

reply = await agent.ask("What is 23.5 plus 8, and then that result multiplied by 3?")
print(reply.body)


First, 23.5 plus 8 equals 31.5. Then, 31.5 multiplied by 3 equals 94.5. So the final result is 94.5.


## Remote servers

Most real MCP servers are remote and need authentication. Same `MCPServer` toolkit — pass an `MCPServerConfig` with a URL instead of an `MCPStdioServerConfig`:

```python
from autogen.beta.tools import MCPServer, MCPServerConfig

agent = Agent(
    "weather_bot",
    prompt="Answer weather questions.",
    config=config,
    tools=[
        MCPServer(
            MCPServerConfig(
                server_url="https://my-mcp-url.example.com",
                authorization_token="XXXXXX",
            )
        )
    ],
)
```

`MCPServerConfig` also accepts `headers`, `allowed_tools`, `blocked_tools`, and a `connection_timeout`. You can register as many `MCPServer` toolkits as you like — and freely mix local and remote ones.

## Additional content

**Provider-side: `MCPServerTool`.** If you only target a provider that supports MCP passthrough (e.g. Anthropic), `MCPServerTool(server_url=..., server_label=..., authorization_token=...)` ships the URL straight to the provider — it connects and executes on its end. No local connection, but your credentials leave your infrastructure.

**Filtering.** `allowed_tools` / `blocked_tools` on any config narrow a chatty server down to just the tools you want the agent to see.

**It's just a toolkit.** Because `MCPServer` is a `Toolkit`, MCP tools sit alongside `@tool` functions, `agent.as_tool()` sub-agents, and builtins in the same `tools=[...]` list. The agent treats them all identically.

## What's next

That closes Group 5 — you can now build and ship complete applications. Group 6 turns to **production concerns**: Episode 24 starts with **telemetry** — seeing exactly what your agents are doing.